# RAG Project - Q&A using LangChain, OpenAI and Pinecone

### Requirements and Environment Setup

In [1]:
pip install -q -r ./requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

# .env files are not included in the repository for security reasons (as specified
# in the .gitignore file) so create your own .env file with the following content:
# OPENAI_API_KEY=""
# PINECONE_API_KEY=""
# LANGCHAIN_API_KEY=""
# LANGCHAIN_TRACING_V2="false"

True

### Loading Documents

In [3]:
# loading PDF, DOCX and TXT files as LangChain Documents
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    if extension == '.pdf':
        from langchain_classic.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain_classic.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        from langchain_classic.document_loaders import TextLoader
        loader = TextLoader(file)
    else:
        print('Document format is not supported!')
        return None

    data = loader.load()
    return data

### Chunking Data

In [5]:
def chunk_data(data, chunk_size=256):
    from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks

### Calculating Cost

In [6]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

### Embedding and Uploading to the Pinecone Vector Database

In [7]:
def insert_or_fetch_embeddings(index_name, chunks):
    """Insert or fetch embeddings from Pinecone."""
    import time
    import pinecone
    from langchain_community.vectorstores import Pinecone as PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import ServerlessSpec

    pc = pinecone.Pinecone()
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)

    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name not in existing_indexes:
        print(f"Creating index {index_name} ... ", end="")

        pc.create_index(
            name=index_name,
            dimension=1536,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)

        print("Ok")

    pinecone_index = pc.Index(index_name)

    vector_store = PineconeVectorStore(
        pinecone_index,
        embeddings,
        text_key="text"
    )

    stats = pinecone_index.describe_index_stats()
    total_vectors = stats.get("total_vector_count", 0)

    if total_vectors == 0:
        print(f"Index {index_name} is empty. Adding documents ... ", end="")
        vector_store.add_documents(chunks)
        print("Ok")
    else:
        print(f"Index {index_name} already has {total_vectors} vectors. Loading embeddings ... Ok")

    return vector_store

In [8]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    
    if index_name == 'all':
        indexes = pc.list_indexes().names()
        print('Deleting all indexes ... ')
        for index in indexes:
            pc.delete_index(index)
        print('OK')
    else:
        print(f'Deleting index {index_name} ...', end='')
        pc.delete_index(index_name)
        print('OK')

### Asking and Getting Answers

In [9]:
def ask_and_get_answer(vector_store, q, k=3):
    from langchain_classic.chains import RetrievalQA
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model='gpt-4o-mini', temperature=1)

    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    
    answer = chain.invoke(q)
    return answer

### Running Code

#### Ask a PDF

In [10]:
data = load_document('../M255/M255 Glossary.pdf')
print(data[0].page_content)
print(data[0].metadata)
print(f'There are {len(data)} pages in the data')
print(f'There are {len(data[0].page_content)} characters on the first page')

Loading ../M255/M255 Glossary.pdf
Mathematics and Computing: Level 2 
M255 Object-oriented programming with Java 
M255 
Glossary 
Copyright ª 2006 The Open University WEB 86853 0 
1.1
{'producer': 'Acrobat Distiller 6.0.1 (Windows)', 'creator': '3B2 Total Publishing System 8.07q/W Unicode', 'creationdate': '2008-09-17T14:10:51+01:00', 'author': 'The Open University', 'moddate': '2008-09-17T15:32:45+01:00', 'title': 'm255 glossary', 'source': '../M255/M255 Glossary.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}
There are 23 pages in the data
There are 149 characters on the first page


In [11]:
chunks = chunk_data(data)
print(len(chunks))
print(chunks[1].page_content)

278
M255 Glossary 2 
A 
abstract class A class that deﬁnes a common message protocol and common set of 
instance variables for its subclasses. In Java an abstract class cannot be 
instantiated.


In [12]:
print_embedding_cost(chunks)

Total Tokens: 11941
Embedding Cost in USD: 0.000239


In [13]:
import pinecone
pc = pinecone.Pinecone()
pc.list_indexes()

IndexList([<name='ou-comp-it-degree', dim=1536, ready=True>])

In [25]:
from pinecone import ServerlessSpec
index_name = 'ou-comp-it-degree'
if index_name not in pc.list_indexes().names():
    print(f'Creating index: {index_name} ...', end='')
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print('Index created!')
else:
    print(f'Index {index_name} already exists!')
# pc.list_indexes()[0]['name']
pc.describe_index('ou-comp-it-degree')

Index ou-comp-it-degree already exists!


IndexModel(name='ou-comp-it-degree', metric='cosine', host='https://ou-comp-it-degree-moalxso.svc.aped-4627-b74a.pinecone.io', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), vector_type='dense', dimension=1536, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [33]:
# Only use this if you need to
# delete_pinecone_index()

Deleting all indexes ... 
OK


In [26]:
index_name = 'ou-comp-it-degree'
index = pc.Index(index_name)
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=278, metric='cosine', namespaces=1)

In [27]:
index_name = 'ou-comp-it-degree'
vector_store = insert_or_fetch_embeddings(index_name=index_name, chunks=chunks)

Index ou-comp-it-degree already has 278 vectors. Loading embeddings ... Ok


In [19]:
q = 'What is an abstract class? Answer from the provided context only.'
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'What is an abstract class? Answer from the provided context only.', 'result': 'An abstract class is a class that defines a common message protocol and a common set of instance variables for its subclasses. In Java, an abstract class cannot be instantiated.'}


#### While Loop for Asking Questions

In [24]:
import time
i = 1
print('Write Quit to quit.')
while True:
    q = input(f'Question #{i}: ') + ' Answer from the provided context only.'
    i = i + 1
    if q.lower().startswith('quit'):
        print('Quitting ... bye bye!')
        time.sleep(2)
        break
    
    answer = ask_and_get_answer(vector_store, q)
    print(f'\nAnswer: {answer}')
    print(f'\n {"-" * 50} \n')



Write Quit to quit.

Answer: {'query': 'What is an abstract class? Answer from the provided context only.', 'result': 'An abstract class is a class that defines a common message protocol and a common set of instance variables for its subclasses. In Java, an abstract class cannot be instantiated.'}

 -------------------------------------------------- 


Answer: {'query': 'What is an access modifier? Answer from the provided context only.', 'result': 'An access modifier is one of three Java keywords (public, private, protected) which specify the visibility of variables and methods.'}

 -------------------------------------------------- 

Quitting ... bye bye!
